# 04 — Unified Evaluation: Three-Paradigm Comparison

**RecSys 2026 Tutorial**: From Retrieval to Generation

This notebook loads results from all three paradigm notebooks and creates
the comparative evaluation tables and visualizations for the tutorial and
the proposal's preliminary results.

**Prerequisite**: Run notebooks 00, 01, 02, and 03 first.

In [ ]:
import sys
sys.path.insert(0, '.')

import json
import pandas as pd
import numpy as np
from tutorial_utils import load_results, RESULTS_DIR

## 1. Load results from all paradigms

In [ ]:
paradigms = ['explainable', 'rag', 'generative']
all_results = {}

for name in paradigms:
    try:
        all_results[name] = load_results(name)
        print(f"Loaded {name}: {all_results[name]['model']}")
    except FileNotFoundError:
        print(f"WARNING: {name} results not found. Run notebook 0{paradigms.index(name)+1} first.")

## 2. Ranking metrics comparison

In [ ]:
ranking_rows = []
for name, res in all_results.items():
    row = {'Paradigm': name.capitalize(), 'Model': res.get('model', '')}
    for metric, val in res.get('ranking', {}).items():
        row[metric] = f"{val:.4f}"
    ranking_rows.append(row)

df_ranking = pd.DataFrame(ranking_rows)
print("=" * 80)
print("Table 1: Ranking Performance on Amazon Books")
print("=" * 80)
print(df_ranking.to_string(index=False))

## 3. Explanation quality comparison

In [ ]:
expl_rows = []
for name, res in all_results.items():
    row = {'Paradigm': name.capitalize()}
    for metric, val in res.get('explanation', {}).items():
        row[metric.replace('_', ' ').title()] = f"{val:.4f}"
    expl_rows.append(row)

df_expl = pd.DataFrame(expl_rows)
print("=" * 80)
print("Table 2: Explanation Quality on Amazon Books")
print("=" * 80)
print(df_expl.to_string(index=False))

## 4. System metrics comparison

In [ ]:
sys_rows = []
for name, res in all_results.items():
    sys_info = res.get('system', {})
    row = {'Paradigm': name.capitalize()}

    # Extract latency from various possible structures
    if 'ranking_latency' in sys_info:
        row['Ranking Latency (ms)'] = f"{sys_info['ranking_latency'].get('mean_latency_ms', 0):.1f}"
    if 'explanation_latency' in sys_info:
        row['Explanation Latency (ms)'] = f"{sys_info['explanation_latency'].get('mean_latency_ms', 0):.1f}"
    if 'latency' in sys_info:
        row['Total Latency (ms)'] = f"{sys_info['latency'].get('mean_latency_ms', 0):.1f}"

    sys_rows.append(row)

df_sys = pd.DataFrame(sys_rows).fillna('N/A')
print("=" * 80)
print("Table 3: System Metrics")
print("=" * 80)
print(df_sys.to_string(index=False))

## 5. Combined summary table (for proposal)

In [ ]:
summary_rows = []
for name, res in all_results.items():
    ranking = res.get('ranking', {})
    expl = res.get('explanation', {})
    summary_rows.append({
        'Paradigm': name.capitalize(),
        'Anchor Paper': res.get('anchor_paper', ''),
        'HR@10': ranking.get('HR@10', 'N/A'),
        'NDCG@10': ranking.get('NDCG@10', 'N/A'),
        'Expl. Relevance': expl.get('relevance', 'N/A'),
        'Hallucination Rate': expl.get('hallucination_rate', 'N/A'),
    })

df_summary = pd.DataFrame(summary_rows)

# Format numeric columns
for col in ['HR@10', 'NDCG@10', 'Expl. Relevance', 'Hallucination Rate']:
    df_summary[col] = df_summary[col].apply(
        lambda x: f"{x:.4f}" if isinstance(x, (int, float)) else str(x)
    )

print("=" * 100)
print("Summary: Three-Paradigm Comparison on Amazon Books")
print("=" * 100)
print(df_summary.to_string(index=False))
print("\n(Use this table for the tutorial proposal preliminary results section)")

## 6. Export for LaTeX

In [ ]:
# Generate LaTeX table for the proposal
latex = df_summary.to_latex(index=False, escape=True, column_format='l' * len(df_summary.columns))
print("LaTeX table (copy into proposal):")
print(latex)